## Multi-Head Attention Mechanism

Welcome to "Deconstructing the Transformer Architecture"! You've just completed an incredible journey through Sequence Models & The Dawn of Attention, where you built the foundational understanding of attention mechanisms and created a standalone PyTorch module for scaled dot-product attention. Now you're ready to take the next major step in your exploration of modern NLP architectures.

In this course, you'll systematically build the complete Transformer architecture from the ground up. You'll start by enhancing your attention mechanism with Multi-Head Attention, then explore positional encodings, layer normalization, and feed-forward networks. By the end, you'll have a fully functional Transformer that can handle real sequence-to-sequence tasks. Our first lesson focuses on Multi-Head Attention, the mechanism that allows Transformers to attend to different types of information simultaneously across multiple representation subspaces.

---

### The Power of Multiple Attention Perspectives

While our single-head attention mechanism from the previous course works well, it has a fundamental limitation: it can only focus on one type of relationship at a time. Imagine reading a sentence where you need to simultaneously track grammatical dependencies, semantic relationships, and contextual references. A single attention head might excel at one of these tasks but struggle to capture all of them effectively.

Multi-Head Attention solves this by running multiple attention computations in parallel, each focusing on different aspects of the input relationships. Think of it as having multiple experts, each specializing in different types of patterns. One head might focus on local dependencies, another on long-range relationships, and yet another on specific semantic patterns. The key insight is that by running these computations in parallel and combining their results, you can capture much richer representations than any single attention mechanism could provide.

The mathematical foundation remains the same scaled dot-product attention you mastered previously, but now you apply it across multiple "heads" simultaneously. Each head operates on different learned projections of the input, allowing the model to attend to information from different representation subspaces at different positions. Mathematically, if you have $h$ heads and model dimension $d_{model}$, each head operates on dimension $d_k = d_{model} / h$, ensuring computational efficiency while maintaining expressiveness.

---

### The Architecture: Linear Projections and Head Division

Let's begin implementing Multi-Head Attention by establishing the core architecture. The first crucial component involves creating separate linear projections for queries, keys, and values, then splitting these projections across multiple attention heads:

```python
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        """
        Multi-Head Attention module

        Args:
            d_model: Model dimension
            num_heads: Number of attention heads
            dropout: Dropout rate
        """
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # Linear projections for Q, K, V and output
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)
        self._init_weights()
```

The constructor establishes several critical design decisions. We ensure `d_model` is divisible by `num_heads` because we'll split the model dimension evenly across heads. Each head operates on $d_k = d_{model} / \text{num\_heads}$ dimensions, meaning if you have a 512-dimensional model with 8 heads, each head works with 64-dimensional projections. The four linear layers create separate learned transformations for queries, keys, values, and the final output projection that combines results from all heads.

The key insight here is that while you maintain the same total model dimension, you're creating multiple parallel pathways for information flow. Each of the projection matrices $W^Q$, $W^K$, and $W^V$ has shape $(d_{model} \times d_{model})$, but their outputs will be reshaped to create multiple heads. The output projection $W^O$ serves a crucial role in learning how to optimally combine information from all heads back into a unified representation.

---

### Implementing Scaled Dot-Product for Multiple Heads

Now you implement the core attention computation that will be applied independently to each head. This method encapsulates the scaled dot-product attention you learned previously, but is designed to handle the multi-head tensor structure:

```python
def scaled_dot_product_attention(self, q, k, v, mask=None):
    """Scaled dot-product attention for multiple heads"""
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)

    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)

    attention_weights = F.softmax(scores, dim=-1)
    attention_weights = self.dropout(attention_weights)

    output = torch.matmul(attention_weights, v)
    return output, attention_weights
```

This implementation mirrors your previous scaled dot-product attention but operates on tensors with an additional head dimension. The key insight is that the same mathematical operations apply regardless of whether you're computing attention for one head or multiple heads simultaneously. The `transpose(-2, -1)` operation and matrix multiplications work seamlessly across the head dimension through broadcasting, allowing you to compute attention for all heads in parallel rather than using expensive loops. The dropout applied to the attention weights serves as a regularization mechanism, randomly zeroing out attention connections during training to prevent the model from over-relying on specific attention patterns and improve generalization.

The attention computation for each head follows the formula:

$$\text{Attention}(Q_i, K_i, V_i) = \text{softmax}\left(\frac{Q_i K_i^T}{\sqrt{d_k}}\right) V_i$$

where $i$ represents the $i$-th attention head. The scaling factor $\sqrt{d_k}$ becomes even more critical in multi-head attention because each head operates on a smaller dimension, making the dot products more prone to extreme values without proper scaling.

---

### Tensor Reshaping and Multi-Head Computation

The heart of Multi-Head Attention lies in how you reshape tensors to create multiple heads and then recombine their outputs. This section handles the complex tensor manipulations that enable parallel attention computation:

```python
def forward(self, query, key, value, mask=None):
    """Forward pass for multi-head attention"""
    batch_size, seq_len_q = query.size(0), query.size(1)
    seq_len_k = key.size(1)

    # Linear projections and reshape for multi-head attention
    Q = self.w_q(query).view(batch_size, seq_len_q, self.num_heads, self.d_k).transpose(1, 2)
    K = self.w_k(key).view(batch_size, seq_len_k, self.num_heads, self.d_k).transpose(1, 2)
    V = self.w_v(value).view(batch_size, seq_len_k, self.num_heads, self.d_k).transpose(1, 2)
```

The reshaping operation is crucial: you transform tensors from `(batch_size, seq_len, d_model)` to `(batch_size, num_heads, seq_len, d_k)`. The `view` operation splits the model dimension into separate heads, while `transpose(1, 2)` moves the head dimension to the second position for efficient computation. This tensor manipulation is what enables the "multi-head" aspect, because you're essentially creating multiple parallel attention computations from a single input.

```python
    # Adjust mask for multi-head attention
    if mask is not None:
        if mask.dim() == 2:  # [seq_len, seq_len]
            mask = mask.unsqueeze(0).unsqueeze(0)  # [1, 1, seq_len, seq_len]
        elif mask.dim() == 3:  # [batch_size, seq_len, seq_len]
            mask = mask.unsqueeze(1)  # [batch_size, 1, seq_len, seq_len]

    # Apply attention
    attention_output, attention_weights = self.scaled_dot_product_attention(Q, K, V, mask)

    # Concatenate heads and apply output projection
    attention_output = attention_output.transpose(1, 2).contiguous().view(
        batch_size, seq_len_q, self.d_model)
    output = self.w_o(attention_output)

    return output, attention_weights
```

The mask handling ensures compatibility with the multi-head structure by adding necessary dimensions for proper broadcasting across all heads. After computing attention, you reverse the reshaping process: transposing back and using `view` to concatenate all head outputs into the original `d_model` dimension. The `contiguous()` call ensures the tensor is stored in a contiguous block of memory, which is required before reshaping. Finally, the output projection `w_o` learns how to best combine information from all heads, implementing the concatenation and linear transformation specified in the original Transformer paper.

---

### Network Initialization

Proper weight initialization and comprehensive testing are crucial for ensuring your Multi-Head Attention module functions correctly. Let's implement the initialization method and create a thorough testing framework:

```python
def _init_weights(self):
    """Initialize weights using Xavier uniform initialization"""
    for linear in [self.w_q, self.w_k, self.w_v, self.w_o]:
        nn.init.xavier_uniform_(linear.weight)
        if linear.bias is not None:
            nn.init.constant_(linear.bias, 0)
```

Xavier uniform initialization helps ensure stable training by preventing vanishing or exploding gradients during the initial training phases. This initialization strategy considers the number of input and output connections to set appropriate initial weight magnitudes.

---

### Testing Our Implementation

Now let's create tests to verify that your Multi-Head Attention implementation works correctly across different scenarios:

```python
def test_multi_head_attention():
    """Test Multi-Head Attention module"""
    print("Testing Multi-Head Attention Implementation...")

    # Configuration
    batch_size = 2
    seq_len = 8
    d_model = 64
    num_heads = 8

    # Create sample input
    torch.manual_seed(42)
    x = torch.randn(batch_size, seq_len, d_model)

    # Initialize Multi-Head Attention
    mha = MultiHeadAttention(d_model, num_heads)

    print(f"Input shape: {x.shape}")
    print(f"Model config: d_model={d_model}, num_heads={num_heads}, d_k={d_model//num_heads}")

    # Test self-attention (Q=K=V)
    output, attention_weights = mha(x, x, x)

    print(f"Output shape: {output.shape}")
    print(f"Attention weights shape: {attention_weights.shape}")

    # Verify dimensions are preserved
    assert output.shape == x.shape, f"Output shape {output.shape} != input shape {x.shape}"

    # Test with causal mask
    causal_mask = torch.tril(torch.ones(seq_len, seq_len))
    masked_output, masked_attn = mha(x, x, x, causal_mask)

    print(f"Masked output shape: {masked_output.shape}")

    # Test gradient flow
    loss = output.mean()
    loss.backward()

    print(f"Gradient verification:")
    for name, param in mha.named_parameters():
        if param.grad is not None:
            print(f"  {name}: grad_norm={param.grad.norm():.6f}")

    total_params = sum(p.numel() for p in mha.parameters())
    print(f"\nTotal parameters: {total_params:,}")
```

This testing framework validates critical aspects of the implementation. You verify that input and output dimensions are preserved, test both unmasked and causal-masked scenarios, and examine gradient flow across all parameters. The self-attention setup (where query, key, and value are all the same input `x`) is fundamental to Transformer architectures and provides a clear test case for your implementation.

---

### Verifying the Results

When you run this complete test, it produces the following comprehensive output that confirms your implementation works correctly:

```text
Testing Multi-Head Attention Implementation...
Input shape: torch.Size([2, 8, 64])
Model config: d_model=64, num_heads=8, d_k=8
Output shape: torch.Size([2, 8, 64])
Attention weights shape: torch.Size([2, 8, 8, 8])
Masked output shape: torch.Size([2, 8, 64])
Gradient verification:
  w_q.weight: grad_norm=0.080937
  w_q.bias: grad_norm=0.023663
  w_k.weight: grad_norm=0.099653
  w_k.bias: grad_norm=0.000000
  w_v.weight: grad_norm=0.201113
  w_v.bias: grad_norm=0.088934
  w_o.weight: grad_norm=0.257533
  w_o.bias: grad_norm=0.125000

Total parameters: 16,640
```

This output confirms several crucial aspects of your implementation: input and output shapes are preserved, attention weights have the expected multi-head structure (batch, heads, seq, seq), and gradient flow is verified across all parameters. The causal mask creates a lower triangular matrix that prevents positions from attending to future positions, essential for autoregressive generation tasks. The total parameter count of 16,640 reflects four projection matrices, each with $64 \times 64 = 4{,}096$ weights plus 64 bias terms, totaling $4 \times (4{,}096 + 64) = 16{,}640$ parameters.

---

### Computational Complexity and Optimization Considerations

Multi-Head Attention, while powerful, comes with significant computational costs that scale quadratically with sequence length. The time complexity is $O(n^2 \cdot d)$ where $n$ is the sequence length and $d$ is the model dimension, dominated by the computation of attention scores between all pairs of positions. Memory complexity follows a similar pattern, requiring $O(n^2)$ storage for attention matrices across all heads. For longer sequences (thousands of tokens), this quadratic scaling becomes prohibitive, making standard attention computationally expensive for tasks like document processing or long-form generation.

Several optimization techniques have emerged to address these limitations. Flash Attention restructures the computation to be more memory-efficient by avoiding the materialization of large attention matrices, achieving significant speedups especially on modern GPUs. Sparse attention patterns limit attention to local windows or specific patterns (like strided or dilated attention), reducing complexity to $O(n \cdot \sqrt{n})$ or $O(n \cdot \log n)$. Other approaches include linear attention approximations that replace the softmax with more efficient operations, gradient checkpointing to trade computation for memory during training, and mixed precision training using FP16 or BF16 to reduce memory footprint while maintaining numerical stability. Understanding these trade-offs becomes crucial when deploying Transformer models in resource-constrained environments or scaling to very long sequences.

---

### Conclusion and Next Steps

Congratulations! You've successfully implemented Multi-Head Attention, a cornerstone mechanism that enables Transformers to capture multiple types of relationships simultaneously. Your implementation demonstrates how parallel attention heads can attend to different representation subspaces, providing a much richer understanding than single-head mechanisms. The careful tensor reshaping and concatenation logic you've mastered form the foundation for more complex Transformer components.

In our next lesson, you'll explore positional encodings, the mechanism that gives Transformers their understanding of sequence order. Unlike RNNs that process sequences step by step, Transformers need explicit positional information, and you'll discover the elegant mathematical solutions that make this possible. Get ready to dive deeper into the architectural innovations that make Transformers so powerful!

## Building Parallel Attention

Now that you've learned about the Multi-Head Attention mechanism, it's time to implement the key dimensional calculation that makes it all work.

---

### Your Task

In the provided code, you'll notice that the `d_k` attribute is set to `None` instead of the correct value. Your job is to calculate the dimension per attention head by dividing the total model dimension (`d_model`) by the number of heads (`num_heads`). This calculation is crucial because it determines how we split the model's representation across multiple attention heads.

Think about it: if we have a model dimension of 512 and 8 attention heads, each head should work with 64 dimensions. This allows each head to focus on different aspects of the input while keeping the total dimensionality consistent.

Once you make this change, the multi-head attention will be able to properly reshape and process the input through its parallel attention mechanisms.

---

### Starter Code

```python
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        """
        Multi-Head Attention module
        
        Args:
            d_model: Model dimension
            num_heads: Number of attention heads
            dropout: Dropout rate
        """
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = None # TODO: Calculate the dimension per head
        
        # Linear projections for Q, K, V and output
        self.w_q = nn.Linear(d_model, d_model) # bias=True by default
        self.w_k = nn.Linear(d_model, d_model) # bias=True by default
        self.w_v = nn.Linear(d_model, d_model) # bias=True by default
        self.w_o = nn.Linear(d_model, d_model) # bias=True by default
        
        self.dropout = nn.Dropout(dropout)
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights using Xavier uniform initialization"""
        for linear in [self.w_q, self.w_k, self.w_v, self.w_o]:
            nn.init.xavier_uniform_(linear.weight)
            if linear.bias is not None:
                nn.init.constant_(linear.bias, 0)
    
    def scaled_dot_product_attention(self, q, k, v, mask=None):
        """Scaled dot-product attention for multiple heads"""
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        
        output = torch.matmul(attention_weights, v)
        return output, attention_weights
    
    def forward(self, query, key, value, mask=None):
        """Forward pass for multi-head attention"""
        batch_size, seq_len_q = query.size(0), query.size(1)
        seq_len_k = key.size(1)
        
        # Linear projections and reshape for multi-head attention
        Q = self.w_q(query).view(batch_size, seq_len_q, self.num_heads, self.d_k).transpose(1, 2)
        K = self.w_k(key).view(batch_size, seq_len_k, self.num_heads, self.d_k).transpose(1, 2)
        V = self.w_v(value).view(batch_size, seq_len_k, self.num_heads, self.d_k).transpose(1, 2)
        
        # Adjust mask for multi-head attention
        if mask is not None:
            # Handle masks without batch dimension
            if mask.dim() == 2:  # [seq_len, seq_len]
                mask = mask.unsqueeze(0).unsqueeze(0)  # [1, 1, seq_len, seq_len]
            elif mask.dim() == 3:  # [batch_size, seq_len, seq_len]
                mask = mask.unsqueeze(1)  # [batch_size, 1, seq_len, seq_len]
            # Broadcasting will automatically handle the heads dimension
        
        # Apply attention
        attention_output, attention_weights = self.scaled_dot_product_attention(Q, K, V, mask)
        
        # Concatenate heads and apply output projection
        attention_output = attention_output.transpose(1, 2).contiguous().view(
            batch_size, seq_len_q, self.d_model)
        output = self.w_o(attention_output)
        
        return output, attention_weights
```

---

### Solution

Only the `d_k` calculation in `__init__` was missing:

```python
self.d_k = d_model // num_heads
```

- `d_model // num_heads` splits the model dimension evenly across the heads, so each head works with $d_k = d_{model} / h$ dimensions. With `d_model = 512` and `num_heads = 8`, that gives `d_k = 64`.
- Integer division (`//`) is the right choice because `d_k` is used as a tensor dimension, and it must be an `int`. The `assert d_model % num_heads == 0` just above guarantees the division is exact.
- `d_k` is used in two places: `forward` reshapes the projections with `.view(batch_size, seq_len, self.num_heads, self.d_k)`, and `scaled_dot_product_attention` scales the scores by `math.sqrt(self.d_k)`.
- Without the fix, `d_k` stays `None`, so the first call to `forward` fails with a `TypeError` in `.view(...)`, and the scaling step would fail with a `TypeError` in `math.sqrt(None)`.

This exercise has no `main()`, so the cell below just defines the full `MultiHeadAttention` class with the fix applied. It can be exercised with a call like `MultiHeadAttention(64, 8)(x, x, x)` on an `x` of shape `(batch_size, seq_len, 64)`.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        """
        Multi-Head Attention module
        
        Args:
            d_model: Model dimension
            num_heads: Number of attention heads
            dropout: Dropout rate
        """
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads # Dimension per head
        
        # Linear projections for Q, K, V and output
        self.w_q = nn.Linear(d_model, d_model) # bias=True by default
        self.w_k = nn.Linear(d_model, d_model) # bias=True by default
        self.w_v = nn.Linear(d_model, d_model) # bias=True by default
        self.w_o = nn.Linear(d_model, d_model) # bias=True by default
        
        self.dropout = nn.Dropout(dropout)
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights using Xavier uniform initialization"""
        for linear in [self.w_q, self.w_k, self.w_v, self.w_o]:
            nn.init.xavier_uniform_(linear.weight)
            if linear.bias is not None:
                nn.init.constant_(linear.bias, 0)
    
    def scaled_dot_product_attention(self, q, k, v, mask=None):
        """Scaled dot-product attention for multiple heads"""
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        
        output = torch.matmul(attention_weights, v)
        return output, attention_weights
    
    def forward(self, query, key, value, mask=None):
        """Forward pass for multi-head attention"""
        batch_size, seq_len_q = query.size(0), query.size(1)
        seq_len_k = key.size(1)
        
        # Linear projections and reshape for multi-head attention
        Q = self.w_q(query).view(batch_size, seq_len_q, self.num_heads, self.d_k).transpose(1, 2)
        K = self.w_k(key).view(batch_size, seq_len_k, self.num_heads, self.d_k).transpose(1, 2)
        V = self.w_v(value).view(batch_size, seq_len_k, self.num_heads, self.d_k).transpose(1, 2)
        
        # Adjust mask for multi-head attention
        if mask is not None:
            # Handle masks without batch dimension
            if mask.dim() == 2:  # [seq_len, seq_len]
                mask = mask.unsqueeze(0).unsqueeze(0)  # [1, 1, seq_len, seq_len]
            elif mask.dim() == 3:  # [batch_size, seq_len, seq_len]
                mask = mask.unsqueeze(1)  # [batch_size, 1, seq_len, seq_len]
            # Broadcasting will automatically handle the heads dimension
        
        # Apply attention
        attention_output, attention_weights = self.scaled_dot_product_attention(Q, K, V, mask)
        
        # Concatenate heads and apply output projection
        attention_output = attention_output.transpose(1, 2).contiguous().view(
            batch_size, seq_len_q, self.d_model)
        output = self.w_o(attention_output)
        
        return output, attention_weights


## Building Strong Neural Foundations

Now that you've calculated the attention head dimensions, let's focus on properly initializing the weights in your multi-head attention module. Good weight initialization is essential for stable training — it prevents gradients from becoming too small or too large.

---

### Your Task

In the given code, you'll notice the `_init_weights` method contains a loop that iterates through all linear layers. Your job is to complete the initialization logic inside this loop by setting up the weights and biases appropriately.

You'll need to use Xavier uniform initialization for the weights and set the biases to zero. Don't forget to check whether the bias exists before trying to initialize it, since some layers might not have biases.

Getting this right will set up your attention mechanism for successful training.

---

### Starter Code

```python
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        """
        Multi-Head Attention module
        
        Args:
            d_model: Model dimension
            num_heads: Number of attention heads
            dropout: Dropout rate
        """
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Linear projections for Q, K, V and output
        self.w_q = nn.Linear(d_model, d_model) # bias=True by default
        self.w_k = nn.Linear(d_model, d_model) # bias=True by default
        self.w_v = nn.Linear(d_model, d_model) # bias=True by default
        self.w_o = nn.Linear(d_model, d_model) # bias=True by default
        
        self.dropout = nn.Dropout(dropout)
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights using Xavier uniform initialization"""
        for linear in [self.w_q, self.w_k, self.w_v, self.w_o]:
            # TODO: Initialize the weights and biases for each linear layer
            pass
    
    def scaled_dot_product_attention(self, q, k, v, mask=None):
        """Scaled dot-product attention for multiple heads"""
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        
        output = torch.matmul(attention_weights, v)
        return output, attention_weights
    
    def forward(self, query, key, value, mask=None):
        """Forward pass for multi-head attention"""
        batch_size, seq_len_q = query.size(0), query.size(1)
        seq_len_k = key.size(1)
        
        # Linear projections and reshape for multi-head attention
        Q = self.w_q(query).view(batch_size, seq_len_q, self.num_heads, self.d_k).transpose(1, 2)
        K = self.w_k(key).view(batch_size, seq_len_k, self.num_heads, self.d_k).transpose(1, 2)
        V = self.w_v(value).view(batch_size, seq_len_k, self.num_heads, self.d_k).transpose(1, 2)
        
        # Adjust mask for multi-head attention
        if mask is not None:
            # Handle masks without batch dimension
            if mask.dim() == 2:  # [seq_len, seq_len]
                mask = mask.unsqueeze(0).unsqueeze(0)  # [1, 1, seq_len, seq_len]
            elif mask.dim() == 3:  # [batch_size, seq_len, seq_len]
                mask = mask.unsqueeze(1)  # [batch_size, 1, seq_len, seq_len]
            # Broadcasting will automatically handle the heads dimension
        
        # Apply attention
        attention_output, attention_weights = self.scaled_dot_product_attention(Q, K, V, mask)
        
        # Concatenate heads and apply output projection
        attention_output = attention_output.transpose(1, 2).contiguous().view(
            batch_size, seq_len_q, self.d_model)
        output = self.w_o(attention_output)
        
        return output, attention_weights
```

---

### Solution

Only the loop body in `_init_weights` was missing:

```python
for linear in [self.w_q, self.w_k, self.w_v, self.w_o]:
    nn.init.xavier_uniform_(linear.weight)
    if linear.bias is not None:
        nn.init.constant_(linear.bias, 0)
```

- `nn.init.xavier_uniform_(linear.weight)` fills each weight matrix with values drawn from a uniform distribution whose range depends on the layer's input and output sizes (its fan-in and fan-out). This keeps the scale of activations and gradients roughly stable across layers, which helps prevent vanishing or exploding gradients early in training.
- `if linear.bias is not None` guards the bias initialization. The four layers here use `nn.Linear`'s default `bias=True`, but a layer created with `bias=False` has `linear.bias` set to `None`, and calling an init function on `None` would raise an error.
- `nn.init.constant_(linear.bias, 0)` sets every bias to zero, so the layers start as pure linear maps and the biases are learned during training.
- The trailing underscore in `xavier_uniform_` and `constant_` marks in-place operations: they modify the parameters directly and return nothing you need to assign.
- Without the fix, the loop body is just `pass`, so the layers keep PyTorch's default `nn.Linear` initialization instead of Xavier uniform weights and zero biases. The code still runs, so the mistake would be easy to miss.

This exercise has no `main()`, so the cell below just defines the full `MultiHeadAttention` class with the fix applied. After constructing it, for example `MultiHeadAttention(64, 8)`, every `w_*.bias` is all zeros.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        """
        Multi-Head Attention module
        
        Args:
            d_model: Model dimension
            num_heads: Number of attention heads
            dropout: Dropout rate
        """
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Linear projections for Q, K, V and output
        self.w_q = nn.Linear(d_model, d_model) # bias=True by default
        self.w_k = nn.Linear(d_model, d_model) # bias=True by default
        self.w_v = nn.Linear(d_model, d_model) # bias=True by default
        self.w_o = nn.Linear(d_model, d_model) # bias=True by default
        
        self.dropout = nn.Dropout(dropout)
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights using Xavier uniform initialization"""
        for linear in [self.w_q, self.w_k, self.w_v, self.w_o]:
            # Xavier uniform for the weights
            nn.init.xavier_uniform_(linear.weight)
            # Zero the bias, but only if the layer has one
            if linear.bias is not None:
                nn.init.constant_(linear.bias, 0)
    
    def scaled_dot_product_attention(self, q, k, v, mask=None):
        """Scaled dot-product attention for multiple heads"""
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        
        output = torch.matmul(attention_weights, v)
        return output, attention_weights
    
    def forward(self, query, key, value, mask=None):
        """Forward pass for multi-head attention"""
        batch_size, seq_len_q = query.size(0), query.size(1)
        seq_len_k = key.size(1)
        
        # Linear projections and reshape for multi-head attention
        Q = self.w_q(query).view(batch_size, seq_len_q, self.num_heads, self.d_k).transpose(1, 2)
        K = self.w_k(key).view(batch_size, seq_len_k, self.num_heads, self.d_k).transpose(1, 2)
        V = self.w_v(value).view(batch_size, seq_len_k, self.num_heads, self.d_k).transpose(1, 2)
        
        # Adjust mask for multi-head attention
        if mask is not None:
            # Handle masks without batch dimension
            if mask.dim() == 2:  # [seq_len, seq_len]
                mask = mask.unsqueeze(0).unsqueeze(0)  # [1, 1, seq_len, seq_len]
            elif mask.dim() == 3:  # [batch_size, seq_len, seq_len]
                mask = mask.unsqueeze(1)  # [batch_size, 1, seq_len, seq_len]
            # Broadcasting will automatically handle the heads dimension
        
        # Apply attention
        attention_output, attention_weights = self.scaled_dot_product_attention(Q, K, V, mask)
        
        # Concatenate heads and apply output projection
        attention_output = attention_output.transpose(1, 2).contiguous().view(
            batch_size, seq_len_q, self.d_model)
        output = self.w_o(attention_output)
        
        return output, attention_weights


## Building Selective Attention Mechanisms

Now that you have proper weight initialization working, it's time to complete the attention masking functionality. Masking is crucial in attention mechanisms — it controls which positions in a sequence can attend to each other, such as preventing future tokens from being seen in causal attention.

---

### Your Task

In the provided code, the `scaled_dot_product_attention` method has incomplete masking logic. Your job is to fill in the missing implementation that applies the mask to the attention scores.

Look for the area where the mask is checked, and add the code that uses `masked_fill` to set scores to a very large negative value (`-1e9`) where the mask indicates positions should be blocked. This ensures those positions receive near-zero attention weights after softmax.

Complete this, and you'll have fully functional masked attention!

---

### Starter Code

```python
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        """
        Multi-Head Attention module
        
        Args:
            d_model: Model dimension
            num_heads: Number of attention heads
            dropout: Dropout rate
        """
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Linear projections for Q, K, V and output
        self.w_q = nn.Linear(d_model, d_model) # bias=True by default
        self.w_k = nn.Linear(d_model, d_model) # bias=True by default
        self.w_v = nn.Linear(d_model, d_model) # bias=True by default
        self.w_o = nn.Linear(d_model, d_model) # bias=True by default
        
        self.dropout = nn.Dropout(dropout)
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights using Xavier uniform initialization"""
        for linear in [self.w_q, self.w_k, self.w_v, self.w_o]:
            nn.init.xavier_uniform_(linear.weight)
            if linear.bias is not None:
                nn.init.constant_(linear.bias, 0)
    
    def scaled_dot_product_attention(self, q, k, v, mask=None):
        """Scaled dot-product attention for multiple heads"""
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if mask is not None:
            # TODO: Apply the mask to the scores tensor using masked_fill
            # Set scores to -1e9 where mask is 0 to prevent attention to those positions
            pass
        
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        
        output = torch.matmul(attention_weights, v)
        return output, attention_weights
    
    def forward(self, query, key, value, mask=None):
        """Forward pass for multi-head attention"""
        batch_size, seq_len_q = query.size(0), query.size(1)
        seq_len_k = key.size(1)
        
        # Linear projections and reshape for multi-head attention
        Q = self.w_q(query).view(batch_size, seq_len_q, self.num_heads, self.d_k).transpose(1, 2)
        K = self.w_k(key).view(batch_size, seq_len_k, self.num_heads, self.d_k).transpose(1, 2)
        V = self.w_v(value).view(batch_size, seq_len_k, self.num_heads, self.d_k).transpose(1, 2)
        
        # Adjust mask for multi-head attention
        if mask is not None:
            # Handle masks without batch dimension
            if mask.dim() == 2:  # [seq_len, seq_len]
                mask = mask.unsqueeze(0).unsqueeze(0)  # [1, 1, seq_len, seq_len]
            elif mask.dim() == 3:  # [batch_size, seq_len, seq_len]
                mask = mask.unsqueeze(1)  # [batch_size, 1, seq_len, seq_len]
            # Broadcasting will automatically handle the heads dimension
        
        # Apply attention
        attention_output, attention_weights = self.scaled_dot_product_attention(Q, K, V, mask)
        
        # Concatenate heads and apply output projection
        attention_output = attention_output.transpose(1, 2).contiguous().view(
            batch_size, seq_len_q, self.d_model)
        output = self.w_o(attention_output)
        
        return output, attention_weights
```

---

### Solution

Only the mask application inside `scaled_dot_product_attention` was missing:

```python
if mask is not None:
    scores = scores.masked_fill(mask == 0, -1e9)
```

- `mask == 0` selects every position the mask marks as blocked, such as future tokens in causal attention. Positions where the mask is `1` (or `True`) are allowed and keep their scores.
- `masked_fill(..., -1e9)` replaces the blocked scores with a very large negative number, so after softmax $e^{-10^9} \approx 0$ and those positions receive essentially zero attention weight. The remaining weights still sum to 1 along each row.
- `masked_fill` returns a new tensor, so the result must be assigned back to `scores`. Calling `scores.masked_fill(...)` without the assignment would leave `scores` unchanged.
- The mask reaches this method already reshaped by `forward` (for example `[1, 1, seq_len, seq_len]` for a 2D causal mask), so it broadcasts across the batch and head dimensions of the `(batch_size, num_heads, seq_len_q, seq_len_k)` scores.
- Without the fix, the `pass` placeholder leaves the scores untouched, so the code runs but masked positions still receive attention, and a causal mask would let each position see future tokens.

This exercise has no `main()`, so the cell below just defines the full `MultiHeadAttention` class with the fix applied. With a causal mask such as `torch.tril(torch.ones(seq_len, seq_len))`, the attention weights above the diagonal come out as zero.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        """
        Multi-Head Attention module
        
        Args:
            d_model: Model dimension
            num_heads: Number of attention heads
            dropout: Dropout rate
        """
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Linear projections for Q, K, V and output
        self.w_q = nn.Linear(d_model, d_model) # bias=True by default
        self.w_k = nn.Linear(d_model, d_model) # bias=True by default
        self.w_v = nn.Linear(d_model, d_model) # bias=True by default
        self.w_o = nn.Linear(d_model, d_model) # bias=True by default
        
        self.dropout = nn.Dropout(dropout)
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights using Xavier uniform initialization"""
        for linear in [self.w_q, self.w_k, self.w_v, self.w_o]:
            nn.init.xavier_uniform_(linear.weight)
            if linear.bias is not None:
                nn.init.constant_(linear.bias, 0)
    
    def scaled_dot_product_attention(self, q, k, v, mask=None):
        """Scaled dot-product attention for multiple heads"""
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if mask is not None:
            # Blocked positions (mask == 0) get -1e9, so softmax makes them ~0
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        
        output = torch.matmul(attention_weights, v)
        return output, attention_weights
    
    def forward(self, query, key, value, mask=None):
        """Forward pass for multi-head attention"""
        batch_size, seq_len_q = query.size(0), query.size(1)
        seq_len_k = key.size(1)
        
        # Linear projections and reshape for multi-head attention
        Q = self.w_q(query).view(batch_size, seq_len_q, self.num_heads, self.d_k).transpose(1, 2)
        K = self.w_k(key).view(batch_size, seq_len_k, self.num_heads, self.d_k).transpose(1, 2)
        V = self.w_v(value).view(batch_size, seq_len_k, self.num_heads, self.d_k).transpose(1, 2)
        
        # Adjust mask for multi-head attention
        if mask is not None:
            # Handle masks without batch dimension
            if mask.dim() == 2:  # [seq_len, seq_len]
                mask = mask.unsqueeze(0).unsqueeze(0)  # [1, 1, seq_len, seq_len]
            elif mask.dim() == 3:  # [batch_size, seq_len, seq_len]
                mask = mask.unsqueeze(1)  # [batch_size, 1, seq_len, seq_len]
            # Broadcasting will automatically handle the heads dimension
        
        # Apply attention
        attention_output, attention_weights = self.scaled_dot_product_attention(Q, K, V, mask)
        
        # Concatenate heads and apply output projection
        attention_output = attention_output.transpose(1, 2).contiguous().view(
            batch_size, seq_len_q, self.d_model)
        output = self.w_o(attention_output)
        
        return output, attention_weights


## Tensor Surgery for Attention Heads

Now that you've mastered attention masking, it's time to tackle the core tensor operations that make multi-head attention work efficiently.

---

### Your Task

In this exercise, you need to complete the reshaping operations for the Query, Key, and Value tensors after they pass through their linear projections. The provided code applies the linear transformations but leaves the reshaping incomplete.

Your job is to add the proper `.view()` and `.transpose()` operations to reshape each tensor. This reshaping is essential because it allows each attention head to process its own slice of the data in parallel.

Complete the reshaping for all three tensors (`Q`, `K`, `V`), and you'll see how multi-head attention splits the computation across multiple heads!

---

### Starter Code

```python
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        """
        Multi-Head Attention module
        
        Args:
            d_model: Model dimension
            num_heads: Number of attention heads
            dropout: Dropout rate
        """
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Linear projections for Q, K, V and output
        self.w_q = nn.Linear(d_model, d_model) # bias=True by default
        self.w_k = nn.Linear(d_model, d_model) # bias=True by default
        self.w_v = nn.Linear(d_model, d_model) # bias=True by default
        self.w_o = nn.Linear(d_model, d_model) # bias=True by default
        
        self.dropout = nn.Dropout(dropout)
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights using Xavier uniform initialization"""
        for linear in [self.w_q, self.w_k, self.w_v, self.w_o]:
            nn.init.xavier_uniform_(linear.weight)
            if linear.bias is not None:
                nn.init.constant_(linear.bias, 0)
    
    def scaled_dot_product_attention(self, q, k, v, mask=None):
        """Scaled dot-product attention for multiple heads"""
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        
        output = torch.matmul(attention_weights, v)
        return output, attention_weights
    
    def forward(self, query, key, value, mask=None):
        """Forward pass for multi-head attention"""
        batch_size, seq_len_q = query.size(0), query.size(1)
        seq_len_k = key.size(1)
        
        # Linear projections and reshape for multi-head attention
        Q = self.w_q(query)  # TODO: Reshape for multi-head
        K = self.w_k(key)    # TODO: Reshape for multi-head
        V = self.w_v(value)  # TODO: Reshape for multi-head
        
        # Adjust mask for multi-head attention
        if mask is not None:
            # Handle masks without batch dimension
            if mask.dim() == 2:  # [seq_len, seq_len]
                mask = mask.unsqueeze(0).unsqueeze(0)  # [1, 1, seq_len, seq_len]
            elif mask.dim() == 3:  # [batch_size, seq_len, seq_len]
                mask = mask.unsqueeze(1)  # [batch_size, 1, seq_len, seq_len]
            # Broadcasting will automatically handle the heads dimension
        
        # Apply attention
        attention_output, attention_weights = self.scaled_dot_product_attention(Q, K, V, mask)
        
        # Concatenate heads and apply output projection
        attention_output = attention_output.transpose(1, 2).contiguous().view(
            batch_size, seq_len_q, self.d_model)
        output = self.w_o(attention_output)
        
        return output, attention_weights
```

---

### Solution

Only the three reshaping lines in `forward` were missing:

```python
Q = self.w_q(query).view(batch_size, seq_len_q, self.num_heads, self.d_k).transpose(1, 2)
K = self.w_k(key).view(batch_size, seq_len_k, self.num_heads, self.d_k).transpose(1, 2)
V = self.w_v(value).view(batch_size, seq_len_k, self.num_heads, self.d_k).transpose(1, 2)
```

- `.view(batch_size, seq_len, self.num_heads, self.d_k)` splits the last dimension of the projection, `d_model`, into `num_heads` slices of `d_k` each, turning `(batch_size, seq_len, d_model)` into `(batch_size, seq_len, num_heads, d_k)`. This works because `d_model = num_heads * d_k`.
- `.transpose(1, 2)` swaps the sequence and head dimensions, giving `(batch_size, num_heads, seq_len, d_k)`. With the head dimension in front of the sequence dimension, the matrix multiplications in `scaled_dot_product_attention` treat each head as its own independent batch item, so all heads run in parallel.
- `Q` uses `seq_len_q`, while `K` and `V` both use `seq_len_k`, since keys and values must have the same sequence length. This also supports cross-attention, where the query and key sequences differ in length.
- Without the fix, `Q`, `K` and `V` stay as the plain 3D projections `(batch_size, seq_len, d_model)` with no head dimension. The rest of `forward` assumes four dimensions: the mask is reshaped to `[1, 1, seq_len, seq_len]` or `[batch_size, 1, seq_len, seq_len]`, and the output is recombined with `transpose(1, 2)` and `view(batch_size, seq_len_q, self.d_model)`. The module therefore no longer computes multi-head attention.

This exercise has no `main()`, so the cell below just defines the full `MultiHeadAttention` class with the fix applied. For an input of shape `(2, 8, 64)` with `num_heads = 8`, `Q`, `K` and `V` become `(2, 8, 8, 8)`, and the attention weights have shape `(2, 8, 8, 8)`, which is `(batch_size, num_heads, seq_len_q, seq_len_k)`.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        """
        Multi-Head Attention module
        
        Args:
            d_model: Model dimension
            num_heads: Number of attention heads
            dropout: Dropout rate
        """
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Linear projections for Q, K, V and output
        self.w_q = nn.Linear(d_model, d_model) # bias=True by default
        self.w_k = nn.Linear(d_model, d_model) # bias=True by default
        self.w_v = nn.Linear(d_model, d_model) # bias=True by default
        self.w_o = nn.Linear(d_model, d_model) # bias=True by default
        
        self.dropout = nn.Dropout(dropout)
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights using Xavier uniform initialization"""
        for linear in [self.w_q, self.w_k, self.w_v, self.w_o]:
            nn.init.xavier_uniform_(linear.weight)
            if linear.bias is not None:
                nn.init.constant_(linear.bias, 0)
    
    def scaled_dot_product_attention(self, q, k, v, mask=None):
        """Scaled dot-product attention for multiple heads"""
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        
        output = torch.matmul(attention_weights, v)
        return output, attention_weights
    
    def forward(self, query, key, value, mask=None):
        """Forward pass for multi-head attention"""
        batch_size, seq_len_q = query.size(0), query.size(1)
        seq_len_k = key.size(1)
        
        # Linear projections and reshape for multi-head attention:
        # (batch, seq, d_model) -> (batch, seq, heads, d_k) -> (batch, heads, seq, d_k)
        Q = self.w_q(query).view(batch_size, seq_len_q, self.num_heads, self.d_k).transpose(1, 2)
        K = self.w_k(key).view(batch_size, seq_len_k, self.num_heads, self.d_k).transpose(1, 2)
        V = self.w_v(value).view(batch_size, seq_len_k, self.num_heads, self.d_k).transpose(1, 2)
        
        # Adjust mask for multi-head attention
        if mask is not None:
            # Handle masks without batch dimension
            if mask.dim() == 2:  # [seq_len, seq_len]
                mask = mask.unsqueeze(0).unsqueeze(0)  # [1, 1, seq_len, seq_len]
            elif mask.dim() == 3:  # [batch_size, seq_len, seq_len]
                mask = mask.unsqueeze(1)  # [batch_size, 1, seq_len, seq_len]
            # Broadcasting will automatically handle the heads dimension
        
        # Apply attention
        attention_output, attention_weights = self.scaled_dot_product_attention(Q, K, V, mask)
        
        # Concatenate heads and apply output projection
        attention_output = attention_output.transpose(1, 2).contiguous().view(
            batch_size, seq_len_q, self.d_model)
        output = self.w_o(attention_output)
        
        return output, attention_weights


## Bringing Attention Heads Back Together